<a href="https://colab.research.google.com/github/rosacarla/Chatbot-baseado-em-regras-e-dados/blob/main/Agente_de_di%C3%A1logo_baseado_em_regras_e_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

># AGENTE DE DI?LOGO H?BRIDO BASEADO EM REGRAS E DADOS
>CURSO: Tecn?logo em Intelig?ncia Artificial Aplicada  
>DISCIPLINA: Agentes Conversacionais  
>AUTORA: Carla Edila Silveira  
>OBJETIVO: construir um agente de di?logo que trar? ocorr?ncias sobre determinado tema  
>MELHORIA: obten??o de dados sobre o Brasil a partir de diferentes fontes  
>GITHUB: https://github.com/rosacarla/Chatbot-baseado-em-regras-e-dados  
>DATA: 05/09/2023
______________________________________________________________________

<body>
<center>
<img src="https://i.postimg.cc/0Q7ZcBm7/header.png" align="middle">
</center>
</body>

>## 1. Qual o contexto do projeto?
><p align="justify">Um agente de di?logo de <i>question answering</i> que, baseado em um corpus de texto sobre um assunto, traz informa??es mais relevantes de acordo com a consulta do usu?rio.</p>

>## 2. Quais ferramentas e t?cnicas adotar?
>*   **NLTK** - toolkit de PLN em Python
>*   **Express?es Regulares** - pacote de regex do Python
>*   **urllib e BeautifulSoup** - bibliotecas para obter dados de p?ginas HTML
>*   **scikit-learn** - pacote com funcionalidades de manipula??o de dados e Machine Learning (ser?o utilizados TF-IDF e Similaridade de cosseno)



> ## 3. Constru??o do agente de di?logo
>A opera??o do agente ser? deste modo:
>1.   Recebe **entrada** do usu?rio
>2.   **Pr?-processa** a entrada do usu?rio
>3.   Calcula a **similaridade** entre a entrada e as senten?as do corpus
>4.   Obt?m a senten?a **mais similar do corpus**
>5.   Mostra-a como **resposta** ao usu?rio  

>Antes destas etapas, ser? criado o corpus ao obter automaticamente dados da Wikipedia.


> ## 4. Importa??o de bibliotecas
> Importar pacote de express?es regulares do Python e acesso ao WordNet dado pelo NLTK.

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('rslp')# Stemming em pt-br
from nltk.corpus import stopwords
nltk.download('stopwords')# Lista de stopwords
import numpy as np
import random
import string
import requests
from bs4 import BeautifulSoup
import bs4 as bs
import urllib.request
import re
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

>## 5. Constru??o do corpus
><p align="justify">Ser? feito um <i>web-scraping</i> para obter os dados automaticamente da Wikipedia. Este processo deve ser executado s? uma vez, e o arquivo salvo em forma de texto na m?quina.</p>

In [ ]:
# Busca paginas sobre o BRASIL. Para mudar o tema basta colocar o link para outra pagina.
# ? possivel obter dados de p?ginas diferentes, basta definir uma lista de links e iterar sobre elas.

# Lista de URLs para consulta
urls = ['https://pt.wikipedia.org/wiki/Brasil',
        'https://brasilescola.uol.com.br/geografia/pais-brasil.htm',
        'https://www.worldbank.org/pt/country/brazil/overview',
        'https://www.bbc.com/portuguese/articles/cw56kkzgnq6o',
        'https://www.bbc.com/portuguese/articles/c3gjpmvkv5eo',
        'https://www.bbc.com/portuguese/articles/c51qdpqxneno',
        'https://www.bbc.com/portuguese/articles/cj79kpjlp59o',
        'https://www.observatoriodasmetropoles.net.br/ibge-apresenta-os-primeiros-resultados-do-censo-2022/'

]

# Inicializa string para armazenar o texto
texto = ''

# Itera sobre URLs
for url in urls:
    try:
        # Solicita HTTP para obter conteudo da pagina
        codigo_html = urllib.request.urlopen(url)

        # Le conteudo da pagina
        codigo_html = codigo_html.read()

        # Processa codigo HTML lido
        html_processado = BeautifulSoup(codigo_html, 'html.parser')

        # Busca todos os paragrafos do texto
        paragrafos = html_processado.find_all('p')

        # Concatena os textos dos paragrafos
        for p in paragrafos:
            texto += p.text + '\n'

    except Exception as e:
        print(f'Erro ao processar a p?gina {url}: {str(e)}')

# Normaliza texto para minusculas
texto = texto.lower()

# Imprime primeiros 1000 caracteres do texto
print(texto[:1000])

>## 6. Pr?-processamento do corpus
> ? necess?rio remover caracteres especiais do texto e divid?-lo em senten?as v?lidas.

In [ ]:
texto = re.sub(r'\[[0-9]*\]', ' ', texto)
texto = re.sub(r'\s+', ' ', texto)
texto[0:1000]

In [ ]:
sentencas = nltk.sent_tokenize(texto, language='portuguese')
palavras = nltk.word_tokenize(texto, language='portuguese')
sentencas[10:15]

>## 7. Fun??es de pr?-processamento de entrada do usu?rio
><p align="justify">Cria fun??es para pr?-processar as entradas do usu?rio, com retirada de  pontua??es e uso de Stemming nos textos, para que palavras similares sejam processadas igualmente pelo algoritmo (por ex., pedra e pedregulho teriam mesma forma l?xica).</p>

In [ ]:
# Instala biblioteca unidecode para eliminar acentuacao de palavras em Python
%pip install unidecode

In [ ]:
# Importa biblioteca unicode
from unidecode import unidecode

In [ ]:
# Define funcao que faz Stemming em todo texto
def stemming(tokens):
  stemmer = nltk.stem.RSLPStemmer()
  novotexto = []
  for token in tokens:
    novotexto.append(stemmer.stem(token.lower()))
  return novotexto

# Funcao que remove pontuacao, stopwords e aplica stemming
def preprocessa(documento):
  # Remove pontuacao
  documento = documento.translate(str.maketrans('', '', string.punctuation))

  # Remove acentos
  documento = unidecode(documento)

  # Tokenizacao de palavras
  tokens = nltk.word_tokenize(documento, language='portuguese')

  # Remove stopwords
  stopwords_pt = set(stopwords.words('portuguese'))
  tokens = [token.lower() for token in tokens if token.lower() not in stopwords_pt]

  # Aplica stemming
  tokens = stemming(tokens)

  return tokens

In [ ]:
# Conferir como fica um texto apos seu pre processamento
preprocessa("Ol? meu nome ? Lucas, eu moro no Brasil, e voc??")

>## 8. Resposta a sauda??es
><p align="justify">Embora seja um sistema de di?logo baseado em tarefas, ? prov?vel que o usu?rio inicie conversas com sauda??es ao agente. Por isso, ser? desenvolvida uma fun??o (regras) para tratar desta situa??o.</p>  
><p align="justify">Ser?o criado grupos de respostas poss?veis, dentre as quais ser?o escolhidas algumas aleatoriamente, para evitar que o agente fique repetitivo.</p>

In [ ]:
saudacoes_entrada = ("ol?", "bom dia", "boa tarde", "boa noite", "oi", "como vai", "e a?", "tudo bem")
saudacoes_respostas = ["ol?", "ol?, espero que esteja tudo bem contigo", "Ol?! Como posso ajudar voc? hoje?", "oi", "Oie",
                       "Seja bem-vindo, em que posso te ajudar?", "Oi! Estou ? disposi??o para responder ?s suas perguntas.",
                       "Ol?! Estou aqui para fornecer informa??es sobre o Brasil. O que voc? gostaria de saber?",
                       "Oi! Bem-vindo! O que voc? deseja saber sobre o Brasil?"]

# Funcao para responder a saudacao
def geradorsaudacoes(saudacao):
    for token in saudacao.split():
        if token.lower() in saudacoes_entrada:
            return random.choice(saudacoes_respostas)

    # Se nenhuma saudacao conhecida for encontrada, retorna None
    return None

In [ ]:
# Ao reexecutar este exemplo serao vistas respostas diferentes
geradorsaudacoes('Ol?')

>## 9. Resposta a consultas do usu?rio
><p align="justify">Fun??o para tratar consultas do usu?rio, com a compara??o da similaridade entre a entrada do usu?rio e as senten?as do corpus. Se houver, a senten?a mais similar ser? mostrada como resposta.</p>
><p align="justify">Nesta fun??o utilizam-se os algoritmos TD-IDF (<i>Term Frequency ? Inverse Document Frequency</i>) e similaridade de cosseno.</p>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Funcao para responder a consultas do usuario
def geradorrespostas(entradausuario): # testar com sentencas
  resposta = ''
  sentencas.append(entradausuario)

  word_vectorizer = TfidfVectorizer(tokenizer=preprocessa, stop_words=stopwords.words('portuguese'))
  all_word_vectors = word_vectorizer.fit_transform(sentencas)
  similar_vector_values = cosine_similarity(all_word_vectors[-1], all_word_vectors)
  similar_sentence_number = similar_vector_values.argsort()[0][-2]

  matched_vector = similar_vector_values.flatten()
  matched_vector.sort()
  vector_matched = matched_vector[-2]

  if vector_matched == 0:
    resposta = resposta + "Me desculpe, n?o entendi o que voc? pediu."
    return resposta
  else:
    resposta = resposta + sentencas[similar_sentence_number]
    return resposta

>## 10. Intera??o com o agente de di?logo
><p align="justify">Define um algoritmo que continue interagindo com o usu?rio at? que ele decida finalizar.</p>  
><p align="justify">O resultado n?o ? sempre o ideal, por?m cobre muitas das poss?veis perguntas. Se utiliz?ssemos apenas regras de di?logo para responder perguntas sobre um assunto, precisar?amos de centenas de regras.</p>
><p align="justify"> Como as respostas s?o baseadas em dados, apenas uma regra que calcula similaridade com o corpus ? suficiente. Fa?a perguntas como:</p>  

>*  *Qual o esporte mais popular no Brasil?*
>*  *Quais eventos esportivos o Brasil j? organizou?*
>*  *Como ? a cozinha brasileira?*
>*  *Onde s?o realizadas pesquisas tecnol?gicas no Brasil?*


In [ ]:
# Inicia interacao com usuario
print("Ol?, eu sou o Agente Tupiniquim. Me pergunte qualquer coisa sobre nosso pa?s ou diga 'tchau' para sair.")
continue_dialogue = True
while continue_dialogue:
  # Obtem entrada do usuario
  human_text = input().lower()

  if human_text != 'tchau':
    if human_text == 'obrigado' or human_text == 'muito obrigado' or human_text == 'agradecido' or human_text == 'valeu':
      continue_dialogue = False
      print("Agente Tupiniquim: Disponha!")
    else:
      if geradorsaudacoes(human_text) != None:
        print("Agente Tupiniquim: " + geradorsaudacoes(human_text))
      else:
        print("Agente Tupiniquim: ", end="")
        print(geradorrespostas(human_text))
        sentencas.remove(human_text)
  else:
    continue_dialogue = False
    print("Agente Tupiniquim: At? a pr?xima!")

>## 11. Como melhorar o projeto?
><p align="justify">Este agente de di?logo adota modelo baseado em regras. Uma das regras usa corpus de dados para formular respostas. Assim, o modelo se torna mais flex?vel, sem necessidade de criar centenas/milhares de regras. Vejamos como melhorar o projeto:</p>  
><p align="justify">A) Al?m dos par?grafos (tag "p") da p?gina da Wikipedia, podem ser usados dados dispostos na coluna direita, que trazem informa??es relevantes, como popula??o, atual presidente etc., para montar senten?as.</p>  
><p align="justify">B) Melhorar o c?lculo de similaridade com uso de um modelo de Word Embeddings, al?m do TF-IDF.</p>
><p align="justify">C) Obter dados sobre o Brasil a partir de diferentes fontes. - OK!</p>
><p align="justify">D) Criar classificador de contexto para o agente e, de modo din?mico, buscar p?ginas da Wikipedia correspondentes ? pergunta do usu?rio, para depois dar a resposta. Desse modo o agente n?o se limitaria a perguntas sobre o Brasil.</p>



>## 12. Refer?ncias e material complementar  
>* [Python for NLP: Creating a Rule-Based Chatbot](https://stackabuse.com/python-for-nlp-creating-a-rule-based-chatbot/)
>* [Building a Simple Chatbot from Scratch in Python (using NLTK)](https://morioh.com/p/6cc33336784c)
>* [Building a simple chatbot in python](https://medium.com/nxtplus/building-a-simple-chatbot-in-python-3963618c490a)
>* [Designing A ChatBot Using Python: A Modified Approach](https://towardsdatascience.com/designing-a-chatbot-using-python-a-modified-approach-96f09fd89c6d)
>* [Build Your First Python Chatbot Project](https://dzone.com/articles/python-chatbot-project-build-your-first-python-pro)
>* [Python Chatbot Project ? Learn to build your first chatbot using NLTK & Keras](https://data-flair.training/blogs/python-chatbot-project/)
>* [Python Chat Bot Tutorial - Chatbot with Deep Learning (Part 1)](https://www.youtube.com/watch?v=wypVcNIH6D4)
>* [Intelligent AI Chatbot in Python](https://www.youtube.com/watch?v=1lwddP0KUEg)
>* [Coding a Jarvis AI Using Python 3 For Beginners](https://www.youtube.com/watch?v=NZMTWBpLUa4)  
>* Projeto elaborado a partir de Notebook criado por Prof. [Lucas Oliveira](http://lattes.cnpq.br/3611246009892500).